<a href="https://colab.research.google.com/github/Le2se0hy/XAI_An/blob/main/%EA%B8%B0%EC%B4%88%ED%86%B5%EA%B3%84%EB%9F%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# ============================================================
# 1. 라이브러리 및 Google Drive 연결
# ============================================================
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount('/content/drive', force_remount=True)

# ============================================================
# 2. 데이터 불러오기
# ============================================================
file_path = '/content/drive/MyDrive/Seoul_Aprtment_FINAL.xlsx'
df = pd.read_excel(file_path)

print("원본 데이터 크기:", df.shape)

# 열 이름 공백과 줄바꿈 정리
df.columns = (
    df.columns
    .astype(str)
    .str.replace('\n', ' ', regex=False)
    .str.replace('\r', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print("\n전체 열 이름:")
print(df.columns.tolist())


# ============================================================
# 3. 연도 열 찾기
# ============================================================
year_candidates = ['Year_Sold']

year_col = None

for col in year_candidates:
    if col in df.columns:
        year_col = col
        break

if year_col is None:
    raise ValueError(
        "연도 열을 찾지 못했습니다. 위에 출력된 전체 열 이름을 확인한 후 "
        "year_col = '실제 연도 열 이름'으로 직접 지정하세요."
    )

print("\n사용할 연도 열:", year_col)


# ============================================================
# 4. 분석할 변수
# ============================================================
variables = [
    'Log_Price_per_m2',
    'Size_m2',
    'Floor',
    'num_of_people',
    'Construction_Year',
    'Parking_per_Household',
    'max_floor',
    'Log_Dist_Water',
    'Log_Dist_Green',
    'Log_Dist_Subway',
    'Dist_CBD',
    'High_School_Count',
    'Bus_Stop',
    'Population',
    'Old Population',
    'Median age',
    'Young Population',
    'Sex_ratio',
    'Pop. Density',
    'Spring',
    'Fall',
    'Winter',
    'heating'
]


# ============================================================
# 5. 변수 존재 여부 확인
# ============================================================
existing_variables = [col for col in variables if col in df.columns]
missing_variables = [col for col in variables if col not in df.columns]

if missing_variables:
    print("\n찾지 못한 변수:")
    for col in missing_variables:
        print("-", col)

print("\n분석에 사용되는 변수 수:", len(existing_variables))


# ============================================================
# 6. 숫자형으로 변환
# ============================================================
df[year_col] = pd.to_numeric(df[year_col], errors='coerce')

for col in existing_variables:
    df[col] = pd.to_numeric(
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.strip(),
        errors='coerce'
    )


# ============================================================
# 7. 연도별 기초통계량 함수
# ============================================================
def make_descriptive_stats(data, year):
    year_data = data[data[year_col] == year].copy()

    stats = (
        year_data[existing_variables]
        .agg(['min', 'max', 'mean', 'std'])
        .T
        .reset_index()
    )

    stats.columns = [
        'Variable',
        'Minimum',
        'Maximum',
        'Mean',
        'Standard Deviation'
    ]

    stats.insert(0, 'Year', year)

    numeric_cols = [
        'Minimum',
        'Maximum',
        'Mean',
        'Standard Deviation'
    ]

    stats[numeric_cols] = stats[numeric_cols].round(4)

    print(f"{year}년 데이터 수:", len(year_data))

    return stats


# ============================================================
# 8. 2022년과 2023년 기초통계량 계산
# ============================================================
stats_2022 = make_descriptive_stats(df, 2022)
stats_2023 = make_descriptive_stats(df, 2023)

print("\n2022년 기초통계량")
display(stats_2022)

print("\n2023년 기초통계량")
display(stats_2023)


# ============================================================
# 9. 두 연도를 하나의 표로 결합
# ============================================================
stats_total = pd.concat(
    [stats_2022, stats_2023],
    ignore_index=True
)

print("\n2022년과 2023년 전체 기초통계량")
display(stats_total)


# ============================================================
# 10. Excel로 저장
# ============================================================
output_path = (
    '/content/drive/MyDrive/'
    'Descriptive_Statistics_2022_2023.xlsx'
)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    stats_2022.to_excel(
        writer,
        sheet_name='2022년',
        index=False
    )

    stats_2023.to_excel(
        writer,
        sheet_name='2023년',
        index=False
    )

    stats_total.to_excel(
        writer,
        sheet_name='2022_2023 통합',
        index=False
    )

print("\n저장 완료:", output_path)

Mounted at /content/drive
원본 데이터 크기: (162375, 41)

전체 열 이름:
['Address_Full', 'Gu_Name', 'Dong', 'Jibun', 'Apartment_Name', 'Price_String', 'Size_m2', 'Floor', 'Construction_Year', 'Year_Sold', 'Month_Sold', 'Day_Sold', 'Price_Won', 'Latitude', 'Longitude', 'High_School_Count', 'Bus_Stop', 'num_of_people', 'heating', 'parking', 'max_floor', 'Parking_per_Household', 'Young Population', 'Median age', 'Old Population', 'Pop. Density', 'Sex_ratio', 'Price_per_m2', 'Log_Price_per_m2', 'Population', 'Dist_Subway', 'Matched_Subway', 'Log_Dist_Subway', 'Dist_Green', 'Matched_Green', 'Log_Dist_Green', 'Dist_Water', 'Matched_Water', 'Log_Dist_Water', 'Dist_CBD', 'Matched_CBD']

사용할 연도 열: Year_Sold

찾지 못한 변수:
- Spring
- Fall
- Winter

분석에 사용되는 변수 수: 20
2022년 데이터 수: 10579
2023년 데이터 수: 31071

2022년 기초통계량


,Year,Variable,Minimum,Maximum,Mean,Standard Deviation
0,2022,Log_Price_per_m2,14.6343,17.8372,16.3321,0.4523
1,2022,Size_m2,11.3300,301.4700,68.5396,38.4536
2,2022,Floor,-1.0000,64.0000,9.5942,6.3731
3,2022,num_of_people,5.0000,6864.0000,880.4406,1172.5004
4,2022,Construction_Year,1970.0000,2024.0000,2004.0630,10.7953
5,2022,Parking_per_Household,0.0270,15.9310,1.1248,1.3176
6,2022,max_floor,4.0000,69.0000,18.2984,8.1753
7,2022,Log_Dist_Water,3.9605,8.5032,6.9514,0.6434
8,2022,Log_Dist_Green,3.8743,7.8506,6.0610,0.5033
9,2022,Log_Dist_Subway,3.7706,8.7987,6.5945,0.5968



2023년 기초통계량


,Year,Variable,Minimum,Maximum,Mean,Standard Deviation
0,2023,Log_Price_per_m2,14.5430,18.0202,16.3582,0.4322
1,2023,Size_m2,10.7800,309.7000,75.1065,29.7603
2,2023,Floor,-3.0000,68.0000,10.0295,6.5868
3,2023,num_of_people,4.0000,6864.0000,1205.0446,1275.3972
4,2023,Construction_Year,1970.0000,2024.0000,2003.2413,11.0610
5,2023,Parking_per_Household,0.0208,13.8028,1.2441,1.3979
6,2023,max_floor,4.0000,69.0000,20.2501,7.8221
7,2023,Log_Dist_Water,4.1345,8.4854,6.9639,0.6214
8,2023,Log_Dist_Green,3.8743,7.7434,6.0683,0.4999
9,2023,Log_Dist_Subway,3.7706,8.7987,6.6595,0.5442



2022년과 2023년 전체 기초통계량


,Year,Variable,Minimum,Maximum,Mean,Standard Deviation
0,2022,Log_Price_per_m2,14.6343,17.8372,16.3321,0.4523
1,2022,Size_m2,11.3300,301.4700,68.5396,38.4536
2,2022,Floor,-1.0000,64.0000,9.5942,6.3731
3,2022,num_of_people,5.0000,6864.0000,880.4406,1172.5004
4,2022,Construction_Year,1970.0000,2024.0000,2004.0630,10.7953
5,2022,Parking_per_Household,0.0270,15.9310,1.1248,1.3176
6,2022,max_floor,4.0000,69.0000,18.2984,8.1753
7,2022,Log_Dist_Water,3.9605,8.5032,6.9514,0.6434
8,2022,Log_Dist_Green,3.8743,7.8506,6.0610,0.5033
9,2022,Log_Dist_Subway,3.7706,8.7987,6.5945,0.5968



저장 완료: /content/drive/MyDrive/Descriptive_Statistics_2022_2023.xlsx


In [11]:
# ============================================================
# 1. 라이브러리 불러오기 및 Google Drive 연결
# ============================================================
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount('/content/drive', force_remount=True)


# ============================================================
# 2. 데이터 불러오기
# ============================================================
file_path = '/content/drive/MyDrive/Seoul_Aprtment_FINAL.xlsx'

df = pd.read_excel(file_path)

print("원본 데이터 크기:", df.shape)


# ============================================================
# 3. 열 이름 정리
# ============================================================
df.columns = (
    df.columns
    .astype(str)
    .str.replace('\n', ' ', regex=False)
    .str.replace('\r', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print("\n전체 열 이름")
print(df.columns.tolist())


# ============================================================
# 4. 연도 열 찾기
# ============================================================
year_candidates = ['Year_Sold']

year_col = None

for col in year_candidates:
    if col in df.columns:
        year_col = col
        break

if year_col is None:
    raise ValueError(
        "연도 열을 찾지 못했습니다.\n"
        "출력된 전체 열 이름을 확인한 후 year_col을 직접 지정하세요."
    )

print("\n사용할 연도 열:", year_col)


# ============================================================
# 5. 필요한 열 확인
# ============================================================
required_columns = [
    year_col,
    'heating',
    'Month_Sold'
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"다음 열을 찾지 못했습니다: {missing_columns}"
    )


# ============================================================
# 6. 연도 열 숫자형 변환
# ============================================================
df[year_col] = pd.to_numeric(
    df[year_col],
    errors='coerce'
)


# ============================================================
# 7. Heating 더미변수 생성
# 정확히 '도시가스'인 경우만 1
# 나머지는 모두 0
# ============================================================
heating_clean = (
    df['heating']
    .fillna('')
    .astype(str)
    .str.strip()
)

df['Heating'] = (
    heating_clean == '도시가스'
).astype(int)


# ============================================================
# 8. month_sold에서 월 추출
# ============================================================
# 3, 4, 5 또는 '3월', '2022-03-01' 형식 모두 고려

month_number = pd.to_numeric(
    df['Month_Sold'],
    errors='coerce'
)

month_from_text = pd.to_numeric(
    df['Month_Sold']
    .astype(str)
    .str.extract(r'(\d{1,2})')[0],
    errors='coerce'
)

month_from_date = pd.to_datetime(
    df['Month_Sold'],
    errors='coerce'
).dt.month

df['_month'] = (
    month_number
    .fillna(month_from_date)
    .fillna(month_from_text)
)


# ============================================================
# 9. 계절 더미변수 생성
# ============================================================
# Spring: 3월, 4월, 5월
df['Spring'] = (
    df['_month'].isin([3, 4, 5])
).astype(int)

# Fall: 9월, 10월, 11월
df['Fall'] = (
    df['_month'].isin([9, 10, 11])
).astype(int)

# Winter: 12월, 1월, 2월
df['Winter'] = (
    df['_month'].isin([12, 1, 2])
).astype(int)


# ============================================================
# 10. 전처리 결과 확인
# ============================================================
print("\nheating 원래 값별 Heating 변환 결과")

heating_check = (
    df.groupby('heating', dropna=False)['Heating']
    .agg(['count', 'min', 'max', 'mean'])
    .reset_index()
)

display(heating_check)


print("\n월별 계절 더미변수 확인")

month_check = (
    df.groupby('_month', dropna=False)[
        ['Spring', 'Fall', 'Winter']
    ]
    .agg(['count', 'mean'])
)

display(month_check)


# ============================================================
# 11. 기초통계량 계산 함수
# ============================================================
dummy_variables = [
    'Heating',
    'Spring',
    'Fall',
    'Winter'
]

def make_descriptive_stats(data, year):

    year_data = data[
        data[year_col] == year
    ].copy()

    if len(year_data) == 0:
        print(f"{year}년 데이터가 없습니다.")

    stats = (
        year_data[dummy_variables]
        .agg(['min', 'max', 'mean', 'std'])
        .T
        .reset_index()
    )

    stats.columns = [
        'Variable',
        'Minimum',
        'Maximum',
        'Mean',
        'Standard Deviation'
    ]

    stats.insert(0, 'Year', year)

    numeric_columns = [
        'Minimum',
        'Maximum',
        'Mean',
        'Standard Deviation'
    ]

    stats[numeric_columns] = (
        stats[numeric_columns]
        .round(4)
    )

    print(f"{year}년 관측치 수: {len(year_data):,}")

    return stats


# ============================================================
# 12. 2022년과 2023년 기초통계량 계산
# ============================================================
stats_2022_dummy = make_descriptive_stats(
    df,
    2022
)

stats_2023_dummy = make_descriptive_stats(
    df,
    2023
)


print("\n2022년 Heating 및 계절 변수 기초통계량")
display(stats_2022_dummy)

print("\n2023년 Heating 및 계절 변수 기초통계량")
display(stats_2023_dummy)


# ============================================================
# 13. 두 연도 표 통합
# ============================================================
stats_dummy_total = pd.concat(
    [
        stats_2022_dummy,
        stats_2023_dummy
    ],
    ignore_index=True
)

print("\n2022년 및 2023년 통합 표")
display(stats_dummy_total)


# ============================================================
# 14. Excel 파일로 저장
# ============================================================
output_path = (
    '/content/drive/MyDrive/'
    'Dummy_Descriptive_Statistics_2022_2023.xlsx'
)

with pd.ExcelWriter(
    output_path,
    engine='openpyxl'
) as writer:

    stats_2022_dummy.to_excel(
        writer,
        sheet_name='2022년',
        index=False
    )

    stats_2023_dummy.to_excel(
        writer,
        sheet_name='2023년',
        index=False
    )

    stats_dummy_total.to_excel(
        writer,
        sheet_name='2022_2023 통합',
        index=False
    )

    heating_check.to_excel(
        writer,
        sheet_name='Heating 변환확인',
        index=False
    )


# 임시 월 변수 삭제
df.drop(
    columns=['_month'],
    inplace=True
)

print("\n저장 완료")
print(output_path)

Mounted at /content/drive
원본 데이터 크기: (162375, 41)

전체 열 이름
['Address_Full', 'Gu_Name', 'Dong', 'Jibun', 'Apartment_Name', 'Price_String', 'Size_m2', 'Floor', 'Construction_Year', 'Year_Sold', 'Month_Sold', 'Day_Sold', 'Price_Won', 'Latitude', 'Longitude', 'High_School_Count', 'Bus_Stop', 'num_of_people', 'heating', 'parking', 'max_floor', 'Parking_per_Household', 'Young Population', 'Median age', 'Old Population', 'Pop. Density', 'Sex_ratio', 'Price_per_m2', 'Log_Price_per_m2', 'Population', 'Dist_Subway', 'Matched_Subway', 'Log_Dist_Subway', 'Dist_Green', 'Matched_Green', 'Log_Dist_Green', 'Dist_Water', 'Matched_Water', 'Log_Dist_Water', 'Dist_CBD', 'Matched_CBD']

사용할 연도 열: Year_Sold

heating 원래 값별 Heating 변환 결과


,heating,count,min,max,mean
0,LPG,2,0,0,0.0
1,개별난방,4,0,0,0.0
2,도시가스,120429,1,1,1.0
3,열병합,41934,0,0,0.0
4,확인불가,6,0,0,0.0



월별 계절 더미변수 확인


Spring        Fall      Winter     
        count mean  count mean  count mean
_month                                    
1        7731  0.0   7731  0.0   7731  1.0
2       11458  0.0  11458  0.0  11458  1.0
3       16859  1.0  16859  0.0  16859  0.0
4       13448  1.0  13448  0.0  13448  0.0
5       16257  1.0  16257  0.0  16257  0.0
6       21750  0.0  21750  0.0  21750  0.0
7       15825  0.0  15825  0.0  15825  0.0
8       13828  0.0  13828  0.0  13828  0.0
9       14423  0.0  14423  1.0  14423  0.0
10      13891  0.0  13891  1.0  13891  0.0
11       8210  0.0   8210  1.0   8210  0.0
12       8695  0.0   8695  0.0   8695  1.0

2022년 관측치 수: 10,579
2023년 관측치 수: 31,071

2022년 Heating 및 계절 변수 기초통계량


...


2023년 Heating 및 계절 변수 기초통계량


,Year,Variable,Minimum,Maximum,Mean,Standard Deviation
0,2023,Heating,0.0,1.0,0.7202,0.4489
1,2023,Spring,0.0,1.0,0.2757,0.4469
2,2023,Fall,0.0,1.0,0.2226,0.4160
3,2023,Winter,0.0,1.0,0.1707,0.3762



2022년 및 2023년 통합 표


,Year,Variable,Minimum,Maximum,Mean,Standard Deviation
0,2022,Heating,0.0,1.0,0.7965,0.4026
1,2022,Spring,0.0,1.0,0.4163,0.4930
2,2022,Fall,0.0,1.0,0.1521,0.3591
3,2022,Winter,0.0,1.0,0.2269,0.4188
4,2023,Heating,0.0,1.0,0.7202,0.4489
5,2023,Spring,0.0,1.0,0.2757,0.4469
6,2023,Fall,0.0,1.0,0.2226,0.4160
7,2023,Winter,0.0,1.0,0.1707,0.3762



저장 완료
/content/drive/MyDrive/Dummy_Descriptive_Statistics_2022_2023.xlsx
